# Longitudinal CT1 synthesis: the end-to-end nets, and CT1 recovered from the enhancement map

Every arm maps this session's FLAIR, T1, T2 (and, for the guided nets, the prior study's
enhancement map) to this session's CT1 in ONE forward call -- the `task: synthesis` runs. All of
them are run on the **same validation slices** with the same prior study, so the tables are paired.

**Two routes to CT1** for every net with an enhancement map S (SBCDLNet, the guided nets):

| route | CT1 estimate | what it trusts |
|---|---|---|
| `out` | the net's output, `x_hat = B z + dc` | the model's own reconstruction of everything |
| `T1+S` | `T1 + S_hat`, the measured T1 plus the decoded enhancement map | the MEASURED T1 for anatomy, the net only for enhancement |

The net's output is its T1 model plus its S, `x_hat = (x_hat - S_hat) + S_hat`, so `T1+S` is the
output with the model's (smoothed) T1 term swapped for the real T1: any anatomical detail the
sparse code smooths away comes back from the measurement. What it can NOT recover is whatever S
does not carry -- S is trained against `(CT1 - T1)_+`, so where CT1 is DARKER than T1 the route
falls back to T1. The reference row `oracle T1+S` (`T1 + (CT1 - T1)_+`, the true one-sided map)
is the best this route can ever do; `T1 (copy)` is the floor.

The UNets have no S. Their rows carry the **implied** enhancement map `(x_hat - T1)_+` instead,
and every row -- either route, any net -- is scored on its implied map the same way
(`S_rel_err`, `S_rms_ratio`), so enhancement is compared like for like.

## Knobs

In [ ]:
import os, sys, math, time
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# NYUMets h5s store canonical-RAS axes, so a raw imshow puts the eyes on the image's RIGHT.
# DISPLAY ONLY -- the stored pixels and every checkpoint are untouched.
from visualization.image import set_display_orient
set_display_orient("radiological")           # or "neurological", or None to switch it off

# ---- trained nets: key -> that run's SAVED config -----------------------------------------------
# The saved config is the authority on model, cond_idx and the guides, so nothing about a run's
# inputs is restated here. Only keys named in ARMS are loaded; a missing run is skipped.
NYU = "trained_nets/nyumets"
SBG = NYU + "/Synth_SBGuidedCDL_NYUMets_CT1_from_all_guide_otherCT1_w%d_sfree%s_dconly_flex_bs64/config.json"
NETS = {
    # bigpurple/nyumets_bs64_array.sbatch, indices 7-11: effective batch 64 + EMA, fit data
    "unet":        NYU + "/Synth_Unet_NYUMets_CT1_from_all_bs64/config.json",
    "sbcdl":       NYU + "/Synth_SBCDLNet_NYUMets_CT1_from_all_sfree_dconly_bs64/config.json",
    "sbg_w0":      SBG % (0, ""), "sbg_w1": SBG % (1, ""), "sbg_w2": SBG % (2, ""),
    # bigpurple/nyumets_followup_array.sbatch
    "unet2m":      NYU + "/Synth_Unet2M_NYUMets_CT1_from_all_bs64/config.json",       # index 1
    "sbg_w0_add":  SBG % (0, "_additive"),                                            # index 4
    "sbg_w0_sw5":  SBG % (0, "_sw5_e500"),                                            # index 5
}
# bigpurple/synth_sweight_sweep.sbatch: SBCDLNet at s_weight 0 / 1 / 2 / 5 / 10 / 20, 500 epochs
SWEEP_W = (0, 1, 2, 5, 10, 20)
NETS.update({"sbcdl_sw%d" % w: NYU + "/sweep_sweight/Synth_SBCDLNet_NYUMets_CT1_from_all_sfree"
             "_dconly_sw%d_e500_bs64/config.json" % w for w in SWEEP_W})

# ---- what to run ---------------------------------------------------------------------------------
# One table row (and figure column) per arm and route: "<name> out", plus "<name> T1+S" for a net
# with an enhancement map.
ARMS = [
    dict(name="unet",          net="unet"),
    dict(name="unet 2M",       net="unet2m"),
    dict(name="sbcdl",         net="sbcdl"),
    dict(name="guided w0",     net="sbg_w0"),
    dict(name="guided w1",     net="sbg_w1"),
    dict(name="guided w2",     net="sbg_w2"),
    dict(name="guided w0 add", net="sbg_w0_add"),
]
# The s_weight sweep instead -- uncomment to compare the six arms on both routes:
# ARMS = [dict(name="sw%d" % w, net="sbcdl_sw%d" % w) for w in SWEEP_W]

REFERENCES = True          # also score "T1 (copy)" and "oracle T1+S" (T1 + the TRUE (CT1 - T1)_+)
S_CLAMP = False            # the T1+S route adds S_hat as decoded (False) or max(S_hat, 0) (True)
REF_ROW = "sbcdl out"      # per-slice differences are taken against this row
USE_EMA = True             # use net_ema.ckpt when a run wrote one (else net.ckpt)

# ---- data -----------------------------------------------------------------------------------------
N_SLICES = 32              # val slices per refresh
BATCH = 8
SEED = 0                   # refresh() bumps this
SLICE_RANGE = (50, 90)     # original slice indices [lo, hi); None = the run's own setting
ENH_Q = 0.98               # enhancement region: top 2% of CT1 - T1 per slice
CHG_Q = 0.98               # interval-change region: top 2% of |CT1 - prior CT1| per slice

# ---- display: FIXED windows, never per-image percentiles ---------------------------------------
VMIN, VMAX = 0.0, 1.0      # img_raw / scales: the brain sits in roughly [0, 1]
S_VMAX = 0.5               # enhancement maps, magma on [0, S_VMAX]
RES_V = 0.25               # residuals, RdBu on [-RES_V, RES_V]
N_SHOW = 4                 # slices drawn in the figures
MASK_DISPLAY = True        # draw every panel inside the loader's T1 & CT1 mask (each row its own)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

## Load the nets

Only `task: synthesis` runs: one call on `[cond..., guides...]`, no schedule, no noise. A net has an
enhancement-map route if it stores `last_S` on its forward (SBCDLNet and SBGuidedGroupCDL with
`s_mode` set); the UNets do not.

In [ ]:
import datasets                                    # noqa: F401  (registers loaders)
from datasets.registry import build_loader
from models import build_model
from training.common import load_ckpt
from training.i2sb import _split_batch
from training.forward_op import fixed_val_subset, enh_region
from training.metrics import compute_metrics


def load_run(path):
    """A trained synthesis run -> everything needed to run it and to rebuild its inputs."""
    with open(path) as f:
        cfg = yaml.safe_load(f)
    if cfg.get("task") != "synthesis":
        raise ValueError(path + ": task " + repr(cfg.get("task")) + " is not synthesis; the bridge "
                         "nets live in notebooks/immap_sb_longitudinal.ipynb")
    # net.ckpt holds the LIVE weights (for resume); net_ema.ckpt the averaged ones, which are
    # what an EMA run reports its validation metrics on.
    sd = cfg["paths"]["save_dir"]
    live = cfg["paths"].get("ckpt") or os.path.join(sd, "net.ckpt")
    ema = os.path.join(sd, "net_ema.ckpt")
    use_ema = USE_EMA and os.path.exists(ema)
    ckpt = ema if use_ema else live
    if not os.path.exists(ckpt):
        raise FileNotFoundError("no checkpoint at " + ckpt)
    net = build_model(cfg).to(DEVICE).eval()
    load_ckpt(ckpt, model=net, device=DEVICE)
    if getattr(net, "attn_backend", None) == "flex" and DEVICE.type == "cuda" \
            and hasattr(net, "compile_flex"):
        net.compile_flex()                # as train.py does at start-up; the fused kernel is GPU-only

    v, t = cfg["data"]["val"], cfg["training"]
    modes = v.get("guide_mode") or "none"
    modes = [modes] if isinstance(modes, str) else list(modes)
    guided = "other_study" in modes
    p = cfg["model"]["params"]
    return dict(cfg=cfg, net=net, weights="ema" if use_ema else "live",
                cond_idx=list(v.get("cond_idx") or []),
                guided=guided,
                guide_idx=list(v.get("guide_idx") or []) if guided else [],
                guide_window=int(v.get("guide_window") or 0) if guided else 0,
                n_guides=int(v.get("n_guides") or 1) if guided else 0,
                guide_slice=v.get("guide_slice"),
                # the loader's guide_enhancement: ONE plane per guide slice, (CT1 - T1)_+ of the
                # prior study, instead of the stored contrasts in guide_idx (ignored then)
                guide_enh=bool(v.get("guide_enhancement")) if guided else False,
                root=v.get("root"),
                x0_idx=int(v.get("x0_idx", 2)), x1_idx=int(v.get("x1_idx", 1)),
                data_range=float(t.get("data_range", 1.0)),
                # residual mode: the net predicts CT1 - X[:, src]; the output adds it back
                residual_src=int(t.get("residual_src_idx", 1)) if t.get("residual_mode") else None,
                # what the loader did to the pixels this net saw; every arm shares ONE loader
                pixels=(v.get("image_key"), tuple(v.get("scales") or ()), v.get("mask_source"),
                        bool(v.get("mask_apply")), v.get("mask_guides")),
                n_in=p.get("in_chans"),            # a Unet2D states its width; unrolled nets don't
                has_S=hasattr(net, "last_S"),
                s_weight=float(t.get("s_weight", 0.0) or 0.0))


RUNS = {}
for key in dict.fromkeys(a["net"] for a in ARMS):
    if key not in NETS:
        raise KeyError("ARMS names net " + repr(key) + ", which is not in NETS")
    try:
        RUNS[key] = r = load_run(NETS[key])
        print("loaded %-12s %-18s %-4s cond=%s  guide=%s  S=%s"
              % (key, type(r["net"]).__name__, r["weights"], r["cond_idx"],
                 ("%s x %d slices" % ("(CT1-T1)+" if r["guide_enh"] else r["guide_idx"],
                                      2 * r["guide_window"] + 1)) if r["guided"] else "none",
                 ("yes, s_weight %g" % r["s_weight"]) if r["has_S"] else "no"))
    except (FileNotFoundError, OSError) as e:
        print("SKIP " + key + ": " + str(e))

arms = [a for a in ARMS if a["net"] in RUNS]
if not arms:
    raise RuntimeError("none of the nets in ARMS could be loaded; nothing to run")
dropped = [a["name"] for a in ARMS if a not in arms]
if dropped:
    print("dropped arms (net unavailable): " + ", ".join(dropped))

# ---- what has to agree for the table to mean anything ------------------------------------------
first = next(iter(RUNS))
ref = RUNS[first]
for k, r in RUNS.items():
    if r["data_range"] != ref["data_range"]:
        raise ValueError(k + ": data_range " + str(r["data_range"]) + " != "
                         + str(ref["data_range"]) + " (" + first + "); the metrics would not be "
                         "comparable")
    if r["pixels"] != ref["pixels"]:
        raise ValueError(k + " was trained on differently prepared pixels than " + first + ": "
                         + str(r["pixels"]) + " vs " + str(ref["pixels"]) + ". One shared loader "
                         "cannot serve both; run them in separate sessions.")
    if (r["x0_idx"], r["x1_idx"]) != (ref["x0_idx"], ref["x1_idx"]):
        raise ValueError(k + ": a different target / T1 pair than " + first)
roots = {k: r["root"] for k, r in RUNS.items()}
if len(set(roots.values())) > 1:
    print("NOTE: these runs were trained on different data roots; every arm is scored on "
          + str(ref["root"]) + " (" + first + "'s)")
data_range = ref["data_range"]
print("\n%d run(s); data_range %g; x%d -> x%d" % (len(RUNS), data_range, ref["x1_idx"], ref["x0_idx"]))

## Data

**One** loader feeds every arm, so all arms see the identical slices, the identical **prior study**
and the identical mask. It is the guided loader even when no arm is guided: the prior CT1 is what
the `rmse_chg` column needs, so the table is restricted to patients with a prior study either way.

| batch key | image |
|---|---|
| `x0` | this session's CT1 -- the target |
| `x1` | this session's T1 -- the anatomy the `T1+S` route keeps |
| `cond` | the union of every run's conditioning contrasts |
| `guide` | the prior study, in the union of every run's guide contrasts, at offsets -W .. +W |

Each run's own input, `[cond..., guides...]`, is carved out of the shared stacks.

In [ ]:
base = dict(ref["cfg"]["data"]["val"])

# the union of every run's conditioning contrasts, in a stable order
ALL_COND = []
for r in RUNS.values():
    for c in r["cond_idx"]:
        if c not in ALL_COND:
            ALL_COND.append(c)

GUIDED = [r for r in RUNS.values() if r["guided"]]
if any(r["n_guides"] != 1 for r in GUIDED):
    raise ValueError("this notebook assumes n_guides=1 (one prior study per sample)")
if any(r["guide_slice"] != "index" for r in GUIDED):
    raise ValueError("guided runs must use guide_slice='index' to be paired by anatomy")

X0, X1 = ref["x0_idx"], ref["x1_idx"]
# an enhancement-map guide is formed from the prior study's T1 and CT1, so both go in the stack;
# the prior CT1 is always there for rmse_chg and the prior-CT1 panel
GUIDE_IDX = sorted({c for r in GUIDED
                    for c in ((r["x1_idx"], r["x0_idx"]) if r["guide_enh"] else r["guide_idx"])}
                   | {X0})
W = max([r["guide_window"] for r in GUIDED], default=0)
NC = len(GUIDE_IDX)
OCT1_CH = W * NC + GUIDE_IDX.index(X0)          # prior CT1 at offset 0 in the shared stack

vcfg = dict(base)
vcfg.update(name="nyumets_guided", cond_idx=ALL_COND, batch_size=BATCH,
            guide_mode="other_study", guide_idx=GUIDE_IDX, guide_window=W,
            guide_as_cond=False, n_guides=1, guide_slice="index",
            guide_enhancement=False,             # the RAW contrasts; guide_for() forms (CT1-T1)+
            x1_source="contrast",                # x1 = THIS session's T1
            deterministic=True, random_flips=False, crop_size=None, center_crop=None)
if SLICE_RANGE is not None:
    vcfg["slice_range"] = list(SLICE_RANGE)
full = build_loader(vcfg, shuffle=False, drop_last=False)


def guide_for(run, guide):
    """The guide planes THIS run was trained on, from the shared flat (B, G, H, W) stack. Plane
    order is slice offset OUTERMOST (-W..+W), contrast innermost (GUIDE_IDX order)."""
    if not run["guided"]:
        return None
    w = run["guide_window"]
    at = lambda d, c: (W + d) * NC + GUIDE_IDX.index(c)
    if run["guide_enh"]:
        # (CT1 - T1)_+ of the prior slice, from planes the loader has already masked -- the same
        # order of operations as the loader's own guide_enhancement
        ct1 = [at(d, run["x0_idx"]) for d in range(-w, w + 1)]
        t1 = [at(d, run["x1_idx"]) for d in range(-w, w + 1)]
        return (guide[:, ct1] - guide[:, t1]).clamp_min(0.0)
    return guide[:, [at(d, c) for d in range(-w, w + 1) for c in run["guide_idx"]]]


def input_for(run, cond, guide):
    """[cond..., guides...]: what the nyumets_synth loader hands this run."""
    parts = [cond[:, [ALL_COND.index(c) for c in run["cond_idx"]]]]
    g = guide_for(run, guide)
    if g is not None:
        parts.append(g)
    return torch.cat(parts, dim=1)


print("%d val slices | shared cond %s | shared guide %s at offsets %+d..%+d (%d planes)"
      % (len(full.dataset), ALL_COND, GUIDE_IDX, -W, W, NC * (2 * W + 1)))
for key, r in RUNS.items():
    per = 1 if r["guide_enh"] else len(r["guide_idx"])
    got = len(r["cond_idx"]) + (per * (2 * r["guide_window"] + 1) if r["guided"] else 0)
    ok = r["n_in"] is None or r["n_in"] == got
    print("  %-12s input channels = %2d%s" % (key, got, "" if ok else
                                               "   <-- MISMATCH: net wants " + str(r["n_in"])))
    if not ok:
        raise ValueError(key + ": the slicing above does not reproduce what it was trained on")

## Running the arms

`run_arms(seed)` draws a random subset of the validation set and runs every arm on it. Which slices
are drawn depends only on `(seed, n_slices)`, never on the arms, so two calls with the same seed are
paired. Per arm it keeps the output and, for a net with an enhancement map, `S_hat`; the `T1+S`
route is formed from those afterwards, so `S_CLAMP` can be flipped without re-running the nets.

In [ ]:
def change_region(x0, oct1, m, q):
    """Per slice: brain pixels where |CT1 - prior CT1| is in the top (1 - q) fraction -- where the
    two studies disagree, so where copying the prior scan fails."""
    d = (x0 - oct1).abs()
    out = torch.zeros_like(m)
    for i in range(x0.shape[0]):
        inb = m[i] > 0.5
        if inb.sum() < 10:
            continue
        thr = torch.quantile(d[i][inb].float(), q)
        out[i] = (inb & (d[i] >= thr)).float()
    return out


def run_arms(seed, n_slices=None, verbose=True, arm_list=None):
    """-> S: the inputs, the regions, and per arm its output ("out") and S_hat ("S")."""
    arm_list = arms if arm_list is None else arm_list
    loader = fixed_val_subset(full, n_slices or N_SLICES, seed)
    keep = {k: [] for k in ("x0", "x1", "oct1", "m", "enh", "chg")}
    outs = {a["name"]: [] for a in arm_list}
    shat = {}                                       # filled only by nets that produce an S
    seconds = {a["name"]: 0.0 for a in arm_list}

    with torch.no_grad():
        for bi, batch in enumerate(loader):
            x0, x1, cond, mask, _, guide = _split_batch(batch, DEVICE)
            if guide is None:
                raise RuntimeError("the loader returned no guide stack; check guide_mode")
            guide = guide.flatten(1, 2) if guide.dim() == 5 else guide     # (B,G,1,H,W) -> (B,G,H,W)
            m = (mask > 0.5).float()
            oct1 = guide[:, OCT1_CH:OCT1_CH + 1]

            for a in arm_list:
                run = RUNS[a["net"]]
                X = input_for(run, cond, guide)
                if run["has_S"]:
                    run["net"].last_S = None             # never read a stale map
                t0 = time.time()
                out = run["net"](X)
                out = out[0] if isinstance(out, (tuple, list)) else out
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                seconds[a["name"]] += time.time() - t0
                out = out.real if torch.is_complex(out) else out
                if run["residual_src"] is not None:
                    out = out + X[:, run["residual_src"]:run["residual_src"] + 1]
                outs[a["name"]].append(out.cpu())
                s = getattr(run["net"], "last_S", None) if run["has_S"] else None
                if s is not None:                        # None: a net built without an S model
                    shat.setdefault(a["name"], []).append((s.real if torch.is_complex(s) else s).cpu())

            keep["x0"].append(x0.cpu()); keep["x1"].append(x1.cpu())
            keep["oct1"].append(oct1.cpu()); keep["m"].append(m.cpu())
            keep["enh"].append(enh_region(x0, x1, m, ENH_Q).cpu())
            keep["chg"].append(change_region(x0, oct1, m, CHG_Q).cpu())
            if verbose:
                print("batch " + str(bi + 1) + "/" + str(len(loader)), end="\r")

    S = {k: torch.cat(v) for k, v in keep.items()}
    S["out"] = {k: torch.cat(v) for k, v in outs.items()}
    S["S"] = {k: torch.cat(v) for k, v in shat.items()}
    S["sec"] = seconds
    return S


def predictions(S, arm_list=None):
    """Every scored row: "<arm> out", "<arm> T1+S" for a net with S, and the reference rows.
    Formed here, not in run_arms, so S_CLAMP applies without re-running the nets."""
    rows = {}
    if REFERENCES:
        rows["T1 (copy)"] = S["x1"]
        rows["oracle T1+S"] = S["x1"] + (S["x0"] - S["x1"]).clamp_min(0.0)
    for a in (arm_list or arms):
        n = a["name"]
        rows[n + " out"] = S["out"][n]
        if n in S["S"]:
            s = S["S"][n].clamp_min(0.0) if S_CLAMP else S["S"][n]
            rows[n + " T1+S"] = S["x1"] + s
    return rows

## Metrics

Pooled over the brain mask.

* `rmse_enh` / `rmse_rest` -- the CT1 error inside and outside the enhancement region (top 2% of
  CT1 - T1). `rmse_enh` is where enhancement detail is won or lost; `T1+S` should not lose on
  `rmse_rest`, where it is the measured T1 plus whatever S puts there.
* `rmse_chg` -- the error where the two studies differ (top 2% of |CT1 - prior CT1|).
* `S_rel_err` -- rms(S_row - (CT1 - T1)_+) / rms((CT1 - T1)_+), with `S_row = (row - T1)_+` the
  enhancement the row implies. 1 = no better than predicting no enhancement. The SAME formula for
  every row, so a UNet, an unrolled net's output and its `T1+S` route compare directly.
* `S_rms_ratio` -- rms(S_row) / rms((CT1 - T1)_+): how much enhancement energy the row has (1 =
  the right amount). Read it WITH `S_rel_err`: energy in the wrong place raises both.

In [ ]:
COLS = ["psnr", "ssim", "rmse", "rmse_enh", "rmse_rest", "rmse_chg", "S_rel_err", "S_rms_ratio"]


def summarize(S, pred):
    x0, x1, m, enh, chg = S["x0"], S["x1"], S["m"], S["enh"], S["chg"]
    e2 = (pred - x0) ** 2
    rm = lambda w: math.sqrt(float((e2 * w).sum()) / max(float(w.sum()), 1.0))
    s_true = (x0 - x1).clamp_min(0.0)
    s_row = (pred - x1).clamp_min(0.0)
    t2 = max(float((s_true ** 2 * m).sum()), 1e-30)
    mets = compute_metrics(x0 * m, pred * m, data_range=data_range, mask=m)
    return {"psnr": float(mets["psnr"]), "ssim": float(mets["ssim"]), "rmse": rm(m),
            "rmse_enh": rm(enh), "rmse_rest": rm(m * (1 - enh)), "rmse_chg": rm(chg),
            "S_rel_err": math.sqrt(float(((s_row - s_true) ** 2 * m).sum()) / t2),
            "S_rms_ratio": math.sqrt(float((s_row ** 2 * m).sum()) / t2)}


def report(S, arm_list=None):
    rows = predictions(S, arm_list)
    res = {k: summarize(S, v) for k, v in rows.items()}
    w = max(len(n) for n in res) + 1
    print(" " * w + " ".join("%11s" % c for c in COLS) + "     (%d slices)" % S["x0"].shape[0])
    for name, r in res.items():
        print(("%-" + str(w) + "s") % name + " ".join("%11.4f" % r[c] for c in COLS))
    return res

## Figures

`show_slices`: T1, the prior CT1, the target CT1, then every row (both routes per net) on the one
fixed window `[VMIN, VMAX]`. `show_S`: the true `(CT1 - T1)_+`, then per arm its decoded `S_hat`
(if it has one) and the enhancement its output implies, `(out - T1)_+`, on `[0, S_VMAX]` -- this is
where a smooth S shows. Both mask each row with its own brain mask when `MASK_DISPLAY`.

In [ ]:
from visualization.image import subplot_images


def _rows_to_show(S, arm_list):
    rows = predictions(S, arm_list)
    return {k: v for k, v in rows.items() if k != "T1 (copy)"}     # T1 is already column one


def show_slices(S, n=None, idx=None, arm_list=None):
    n = N_SHOW if n is None else n
    idx = list(range(min(n, S["x0"].shape[0]))) if idx is None else list(idx)
    rows_ = _rows_to_show(S, arm_list)
    rows, labels = [], []
    for i in idx:
        mi = S["m"][i, 0] if MASK_DISPLAY else 1.0
        rows.append([S["x1"][i, 0] * mi, S["oct1"][i, 0] * mi, S["x0"][i, 0] * mi]
                    + [v[i, 0] * mi for v in rows_.values()])
        labels.append("slice " + str(i))
    subplot_images(rows, row_labels=labels,
                   col_titles=["T1", "prior CT1", "CT1 (target)"] + list(rows_),
                   cmap="gray", vmin=VMIN, vmax=VMAX, magnitude=False,
                   panel_size=(2.4, 2.6), show=False)
    plt.show()


def show_S(S, n=None, idx=None, arm_list=None):
    n = N_SHOW if n is None else n
    idx = list(range(min(n, S["x0"].shape[0]))) if idx is None else list(idx)
    names = [a["name"] for a in (arm_list or arms)]
    rows, labels = [], []
    titles = ["(CT1 - T1)+ true"]
    for k in names:
        if k in S["S"]:
            titles.append(k + " S_hat")
        titles.append(k + " (out - T1)+")
    for i in idx:
        mi = S["m"][i, 0] if MASK_DISPLAY else 1.0
        row = [(S["x0"][i, 0] - S["x1"][i, 0]).clamp_min(0.0) * mi]
        for k in names:
            if k in S["S"]:
                row.append(S["S"][k][i, 0] * mi)
            row.append((S["out"][k][i, 0] - S["x1"][i, 0]).clamp_min(0.0) * mi)
        rows.append(row)
        labels.append("slice " + str(i))
    subplot_images(rows, row_labels=labels, col_titles=titles, cmap="magma",
                   vmin=0.0, vmax=S_VMAX, magnitude=False, panel_size=(2.4, 2.6), show=False)
    plt.show()

## Refresh

**Re-run this cell for a new random set of validation slices.** `refresh()` bumps the seed each
time; `refresh(seed=...)` returns to a specific draw and `refresh(n_slices=...)` changes how many.
It leaves `S`, `results`, `cur_seed` and `cur_n` in the namespace for the cells below.

In [ ]:
_refresh_n = 0


def refresh(seed=None, n_slices=None, show=True):
    """New slices -> run every arm -> print the table and draw the figures."""
    global S, results, cur_seed, cur_n, _refresh_n
    if seed is None:
        _refresh_n += 1
        seed = SEED + _refresh_n
    cur_seed, cur_n = seed, n_slices
    t0 = time.time()
    S = run_arms(seed, n_slices)
    print("seed " + str(seed) + " | " + str(S["x0"].shape[0]) + " slices | "
          + "%.1f s total" % (time.time() - t0) + " " * 20)
    results = report(S)
    if show:
        show_slices(S)
        show_S(S)
    # DELIBERATELY returns nothing: a cell whose last expression is a dict of image tensors gets
    # it auto-displayed by Jupyter.


refresh(seed=SEED)

## Residuals

Every row's error against CT1 on one fixed diverging window `[-RES_V, RES_V]`, then the
interval-change map on `[0, RES_V]`. For a net with S, the `out` and `T1+S` columns side by side
show where swapping in the measured T1 helps (anatomy) and where it cannot (CT1 darker than T1).

In [ ]:
def show_residuals(S, n=None, idx=None, arm_list=None):
    n = N_SHOW if n is None else n
    idx = list(range(min(n, S["x0"].shape[0]))) if idx is None else list(idx)
    rows_ = _rows_to_show(S, arm_list)
    rows, labels = [], []
    for i in idx:
        mi = S["m"][i, 0] if MASK_DISPLAY else 1.0
        rows.append([(v[i, 0] - S["x0"][i, 0]) * mi for v in rows_.values()]
                    + [(S["x0"][i, 0] - S["oct1"][i, 0]).abs() * mi])
        labels.append("slice " + str(i))
    k = len(rows_)
    subplot_images(rows, row_labels=labels,
                   col_titles=[r + " - CT1" for r in rows_] + ["|CT1 - prior CT1|"],
                   cmap=["RdBu_r"] * k + ["magma"],
                   vmin=[-RES_V] * k + [0.0], vmax=[RES_V] * k + [RES_V],
                   share_window=False, magnitude=False, panel_size=(2.4, 2.6), show=False)
    plt.show()


show_residuals(S)

## Per slice: does `T1+S` beat the net's own output?

For every net with an enhancement map, per-slice `T1+S` minus `out`, in PSNR (positive = the
route wins) and in `rmse_enh` (negative = the route wins inside the enhancement region). Mass on
the winning side of zero on most slices means the gain is systematic, not a few slices.

In [ ]:
from visualization.hist import plot_hist


def per_slice(S, pred, what):
    out = []
    for i in range(S["x0"].shape[0]):
        m = S["m"][i:i + 1]
        if what == "psnr":
            if float(m.sum()) == 0:
                out.append(np.nan)
                continue
            out.append(float(compute_metrics(S["x0"][i:i + 1] * m, pred[i:i + 1] * m,
                                             psnr_only=True, data_range=data_range,
                                             mask=m)["psnr"]))
        else:                                    # rmse inside this slice's enhancement region
            e = S["enh"][i:i + 1]
            out.append(math.sqrt(float((((pred[i:i + 1] - S["x0"][i:i + 1]) ** 2) * e).sum())
                                 / float(e.sum())) if float(e.sum()) > 0 else np.nan)
    return np.array(out)


rows_all = predictions(S)
with_S = [a["name"] for a in arms if a["name"] in S["S"]]
if not with_S:
    print("no loaded arm has an enhancement map; nothing to compare")
else:
    for what, better in (("psnr", "> 0"), ("rmse_enh", "< 0")):
        series = {}
        for k in with_S:
            d = per_slice(S, rows_all[k + " T1+S"], what) - per_slice(S, rows_all[k + " out"], what)
            d = d[np.isfinite(d)]
            series[k] = d
            wins = (d > 0) if what == "psnr" else (d < 0)
            print("%-16s %-8s T1+S - out: mean %+.4f, median %+.4f, route wins on %3.0f%% of %d slices"
                  % (k, what, d.mean(), np.median(d), 100 * wins.mean(), d.size))
        plot_hist(series, bins=30, vlines={"no change": 0.0},
                  xlabel=what + "(T1+S) - " + what + "(out)   (route wins where " + better + ")",
                  title="per slice, " + what, show=True);

## Reading it

1. **`T1+S` against `out`, per net.** The question this notebook was written for. If the route
   wins on `psnr` and `rmse_rest`, the model's own T1 term is the smoothing culprit and the
   measured T1 restores it. If it also wins on `rmse_enh`, S carries enhancement detail the output
   was losing on top of it.
2. **Against `oracle T1+S`.** The ceiling of the route: the TRUE one-sided map added to T1. The gap
   between a net's `T1+S` and the oracle is what S gets wrong; the gap between the oracle and a
   perfect score is what the route cannot represent at all (CT1 darker than T1, and any global
   T1 / CT1 intensity difference).
3. **`S_rel_err` across rows.** For the UNets it is computed from `(out - T1)_+`; for the unrolled
   nets on both routes. It puts "the UNet predicts enhancement better visually" on one number.
4. **The sweep.** Set `ARMS` to the sweep line in the Knobs cell. A larger `s_weight` should lower
   `S_rel_err` on the `T1+S` rows first -- that route reads S directly.

Diagnostics:

* **`T1+S` is worse everywhere, including `rmse_rest`.** S has energy outside the enhancement --
  it is not an enhancement map. Check `S_rms_ratio` and set `S_CLAMP = True` to see how much of it
  is negative.
* **The oracle row is far from perfect.** The one-sided target throws away a real part of CT1 -
  T1 on this data, so even a perfect S leaves an error floor; a signed S would be the next step.
* **`T1+S` wins on `rmse_rest` but loses on `rmse_enh`.** S is too smooth to beat the output inside
  lesions, though the measured T1 helps elsewhere. That is the case for a sharper S (higher
  `s_weight`, or the additive guide blend for the guided nets).